In [1]:
# LSTM CUSTOMER REVIEW SENTIMENT ANALYZER

import pandas as pd
import numpy as np
import re

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Load the IMDb 50K Review Dataset
data = pd.read_csv("IMDB Dataset.csv")

print(data.head())

print("\nDataset Size:")
print(data.shape)

# 2. Clean and Tokenize Reviews
def preprocess(text):
    text = text.lower()
    # Remove HTML tags
    text = re.sub(
        r'<.*?>',
        ' ',
        text
    )

    # Remove special characters and numbers
    text = re.sub(
        r'[^a-zA-Z\s]',
        ' ',
        text
    )

    # Tokenize
    words = text.split()
    return words

data["tokens"] = data["review"].apply(
    preprocess
)

# Convert sentiment labels
# positive = 1
# negative = 0
data["label"] = data["sentiment"].map({
    "positive": 1,
    "negative": 0
})

# 3. Create Vocabulary and Convert Words into Integer Sequences
word_count = {}
for tokens in data["tokens"]:

    for word in tokens:

        word_count[word] = (
            word_count.get(word, 0) + 1
        )
# Keep words appearing at least 2 times

vocab_words = [
    word
    for word, count
    in word_count.items()

    if count >= 2
]

# Special tokens

word_to_index = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in vocab_words:

    word_to_index[word] = (
        len(word_to_index)
    )

vocab_size = len(word_to_index)

print("\nVocabulary Size:")
print(vocab_size)

# Convert words into integer sequences

sequences = []

for tokens in data["tokens"]:
    sequence = [
        word_to_index.get(
            word,
            word_to_index["<UNK>"]
        )

        for word in tokens
    ]

    sequences.append(sequence)

# 4. Pad Sequences to a Fixed Length
MAX_LENGTH = 200
def pad_sequence(sequence):

    if len(sequence) > MAX_LENGTH:

        return sequence[
            :MAX_LENGTH
        ]
    else:
        return sequence + [

            word_to_index["<PAD>"]

        ] * (
            MAX_LENGTH - len(sequence)
        )

padded_sequences = np.array([

    pad_sequence(sequence)

    for sequence in sequences

])

labels = data["label"].values

# 5. Split Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

# Split training data into training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

print("\nTraining Samples:",
      len(X_train))

print("Validation Samples:",
      len(X_val))

print("Testing Samples:",
      len(X_test))

# Convert to PyTorch tensors

X_train = torch.tensor(
    X_train,
    dtype=torch.long
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

X_val = torch.tensor(
    X_val,
    dtype=torch.long
)

y_val = torch.tensor(
    y_val,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.long
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

# DataLoaders

train_dataset = TensorDataset(
    X_train,
    y_train
)

val_dataset = TensorDataset(
    X_val,
    y_val
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64
)


# 6 & 7. Build LSTM Model Embedding + LSTM + Fully Connected
class SentimentLSTM(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=128
    ):

        super().__init__()
        # Embedding Layer

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # LSTM Layer

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Fully Connected Layer

        self.fc = nn.Linear(
            hidden_dim,
            1
        )

    def forward(self, x):
        # Embedding
        embedded = self.embedding(x)

        # LSTM

        output, (hidden, cell) = (
            self.lstm(embedded)
        )

        # Last hidden state
        hidden = hidden[-1]

        # Fully Connected layer
        output = self.fc(hidden)
        return output.squeeze(1)

# Create model
model = SentimentLSTM(
    vocab_size
)

# Loss Function and Optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# 8. Train the Model
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for inputs, targets in train_loader:
        optimizer.zero_grad()

        outputs = model(
            inputs
        )
        loss = criterion(
            outputs,
            targets
        )

        loss.backward()
        optimizer.step()

        total_train_loss += (
            loss.item()
        )

    # Validation
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for inputs, targets in val_loader:

            outputs = model(
                inputs
            )

            loss = criterion(
                outputs,
                targets
            )

            total_val_loss += (
                loss.item()
            )

    train_loss = (
        total_train_loss /
        len(train_loader)
    )

    val_loss = (
        total_val_loss /
        len(val_loader)
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"- Training Loss: {train_loss:.4f} "
        f"- Validation Loss: {val_loss:.4f}"
    )

# 9. Evaluate the Model Accuracy, Precision, Recall, F1-score
model.eval()

all_predictions = []
all_actual = []

with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(
            inputs
        )

        probabilities = torch.sigmoid(
            outputs
        )

        predictions = (
            probabilities >= 0.5
        ).int()

        all_predictions.extend(
            predictions.numpy()
        )

        all_actual.extend(
            targets.numpy().astype(int)
        )

accuracy = accuracy_score(
    all_actual,
    all_predictions
)

precision = precision_score(
    all_actual,
    all_predictions,
    zero_division=0
)

recall = recall_score(
    all_actual,
    all_predictions,
    zero_division=0
)

f1 = f1_score(
    all_actual,
    all_predictions,
    zero_division=0
)

print("\n================================")
print("MODEL EVALUATION")
print("================================")

print(
    "Accuracy:",
    round(accuracy, 4)
)

print(
    "Precision:",
    round(precision, 4)
)

print(
    "Recall:",
    round(recall, 4)
)

print(
    "F1 Score:",
    round(f1, 4)
)

# 10. Predict Sentiment for a New Review
def predict_sentiment(review):

    # Clean and tokenize
    tokens = preprocess(
        review
    )
    # Convert to integers
    sequence = [

        word_to_index.get(
            word,
            word_to_index["<UNK>"]
        )

        for word in tokens
    ]
    # Pad

    sequence = pad_sequence(
        sequence
    )
    # Convert to tensor

    input_tensor = torch.tensor(
        [sequence],
        dtype=torch.long
    )

    model.eval()

    with torch.no_grad():

        output = model(
            input_tensor
        )

        probability = torch.sigmoid(
            output
        ).item()

    # Sentiment

    if probability >= 0.5:
        sentiment = "Positive"
        confidence = probability
    else:
        sentiment = "Negative"
        confidence = 1 - probability

    print("\nPrediction")
    print("----------------")

    print(
        "Review:",
        review
    )

    print(
        "Predicted Sentiment:",
        sentiment
    )

    print(
        "Confidence:",
        round(
            confidence * 100,
            2
        ),
        "%"
    )

# Test New Review
new_review = (
    "The movie was excellent and very enjoyable"
)

predict_sentiment(
    new_review
)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset Size:
(50000, 2)

Vocabulary Size:
62126

Training Samples: 36000
Validation Samples: 4000
Testing Samples: 10000
Epoch 1/5 - Training Loss: 0.6907 - Validation Loss: 0.6865
Epoch 2/5 - Training Loss: 0.6420 - Validation Loss: 0.6454
Epoch 3/5 - Training Loss: 0.5946 - Validation Loss: 0.5845
Epoch 4/5 - Training Loss: 0.5601 - Validation Loss: 0.6544
Epoch 5/5 - Training Loss: 0.4914 - Validation Loss: 0.6835

MODEL EVALUATION
Accuracy: 0.5639
Precision: 0.5446
Recall: 0.781
F1 Score: 0.6417

Prediction
----------------
Review: The movie was excellent and very enjoyable
Predicted Sentiment: Positive
Confi

In [2]:
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.
